# LLM-MedQA: FastEmbed GPU API
Cell 1: Install. Cell 2: Server (cell stays running - DO NOT STOP IT).

In [ ]:
!pip install -q fastapi uvicorn pydantic fastembed-gpu
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print('Done.')

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastembed import TextEmbedding
import threading, subprocess, re, uvicorn

MODEL_NAME = "BAAI/bge-small-en-v1.5"
print(f"Loading {MODEL_NAME}...")
embedder = TextEmbedding(model_name=MODEL_NAME, providers=["CUDAExecutionProvider"])
list(embedder.embed(["warmup"]))
print("Model ready!")

app = FastAPI()

class EmbedRequest(BaseModel):
    texts: list[str]

@app.post("/embed")
def embed_batch(req: EmbedRequest):
    return {"embeddings": [e.tolist() for e in embedder.embed(req.texts)]}

@app.get("/health")
def health():
    return {"status": "ok"}

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
threading.Thread(target=uvicorn.Server(config).run).start()

import time; time.sleep(2)
print("Server up!")

tun = subprocess.Popen(
    ['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in iter(tun.stdout.readline, ''):
    m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if m:
        print('=' * 60)
        print(f'URL: {m.group(0)}')
        print('=' * 60)
        print('Paste URL to local. DO NOT stop this cell.')
        break

# Keep cell alive by waiting for tunnel process (system-level block)
tun.wait()